# M1 Lab 2, First Eval (LLM-as-a-Judge)

Runs Version A (Concise) vs Version B (Narrative) on the Q4 Marketing email, then has a judge model score both, and a 20-row starter dataset, against the golden-set criteria in `eval-harness-proof.md`.

**Setup:** `OPENAI_API_KEY=...` in `.env` at the repo root (gitignored). Deps live in the repo `.venv` (`openai`, `python-dotenv`), created with `uv venv --python 3.14 .venv && uv pip install openai python-dotenv jupyter`.

**Models:** generator `gpt-4.1-nano` (OpenAI, small and cheap), judge `claude-opus-5` (Anthropic, more capable than what it grades and a different model family, which avoids self-preference bias). Runs 1 and 2 used `gpt-5.5` as the judge; all three runs are in `runs/`, outputs verbatim.

Adapted from the instructor's `eval_lab.ipynb` (Gemini): same email, same system prompts, same A-vs-B judge pattern; the provider, the rubric and the dataset cells are mine.

In [1]:
import os, json, csv
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "01-evaluation-strategy" else Path.cwd()
load_dotenv(REPO / ".env")            # OPENAI_API_KEY and ANTHROPIC_API_KEY
openai_client = OpenAI()
claude = anthropic.Anthropic()

GENERATOR_MODEL = "gpt-4.1-nano"      # OpenAI
JUDGE_MODEL = "claude-opus-5"         # Anthropic: different family from the generator
FIXTURES = REPO / "01-evaluation-strategy" / "fixtures"

In [2]:
EMAIL = """Summarize this email for the CEO.

Subject: Q4 Marketing Campaign Update

Hi team,

Social media engagement is up 25% month-over-month due to the new content calendar and paid ad experiments.
Newsletter open rate dropped from 42% to 35%; we'll test new subject lines next week.
The product video campaign is delayed until November, pending legal approval; creative assets are ready.
Please ensure all campaign tracking links are added by Friday.

Best,
Priya"""

SYSTEM_A = (
    "You are an executive briefing assistant.\n"
    "Summarize in exactly 3 bullet points under 60 words. No preamble, no extra text."
)

SYSTEM_B = (
    "You are a PR communications assistant.\n"
    "Write a 100-word narrative summary highlighting wins first, then risks and next steps. "
    "Keep a positive tone. No bullets."
)

In [3]:
def generate(system_prompt: str, user_message: str) -> str:
    r = openai_client.chat.completions.create(
        model=GENERATOR_MODEL,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_message}],
    )
    return r.choices[0].message.content.strip()


output_a = generate(SYSTEM_A, EMAIL)
output_b = generate(SYSTEM_B, EMAIL)

print("=== Version A (Concise) ===")
print(output_a, f"\n({len(output_a.split())} words)")
print("\n=== Version B (Narrative) ===")
print(output_b, f"\n({len(output_b.split())} words)")

=== Version A (Concise) ===
- Social media engagement increased 25% MoM thanks to new content and ad tests.  
- Newsletter open rate declined to 35%; testing new subject lines scheduled.  
- Product video campaign delayed until November; tracking links due Friday. 
(37 words)

=== Version B (Narrative) ===
The Q4 marketing efforts continue to show promising results, with social media engagement increasing by 25% month-over-month thanks to the new content calendar and successful ad experiments. While the newsletter open rate experienced a decline from 42% to 35%, plans are underway to test new subject lines to boost open rates. The product video campaign is on track to launch in November once legal approval is secured, with all creative assets prepared. Moving forward, ensuring that all campaign tracking links are added by Friday will help monitor ongoing progress. Overall, these initiatives position us well for sustained growth in the coming months. 
(102 words)


In [4]:
GOLDEN_CRITERIA = """The reader is a CEO who acts on the summary without opening the email. A summary is judged in two ordered tests.

TEST 1, FIDELITY (a fail here is a fail). The summary is BAD if it:
- drops or changes any number or date in the email (a baseline figure counts);
- drops or contradicts a blocker or dependency. A blocker counts as preserved when it is named at the level the CEO would act on ("legal review may delay launch" preserves "legal review of the revised claims language could delay launch if comments come back late"); its qualifying condition is optional detail, not a required fact;
- adds anything the email does not say;
- softens or upgrades a risk (a 7-point drop is not "a slight dip"; "delayed" is not "moving closer to rollout").

TEST 2, BREVITY (only for summaries that pass Test 1). GOOD means under 60 words, the deadline or ask stated explicitly, and nothing the CEO cannot act on. Style is free: bullets or prose, either is fine.

What a summary must never do: make the CEO feel better informed than they are. A short summary that drops the blocker and a polished one that softens the risk fail for the same reason."""

JUDGE_PROMPT = f"""You are grading two candidate summaries of the same source email against a strict rubric.
Judge each summary independently, then declare a winner.

RUBRIC:
{GOLDEN_CRITERIA}

SOURCE EMAIL:
{EMAIL}

CANDIDATE A:
{output_a}

CANDIDATE B:
{output_b}

For each candidate: list every fidelity violation (quote the summary text and the email text it conflicts with), state whether it passes Test 1, and if so whether it passes Test 2.
Then output a final line exactly in the form: WINNER: A or WINNER: B or WINNER: TIE"""

verdict_msg = claude.messages.create(
    model=JUDGE_MODEL,
    max_tokens=16000,
    messages=[{"role": "user", "content": JUDGE_PROMPT}],
)
verdict = "".join(b.text for b in verdict_msg.content if b.type == "text").strip()
print(verdict)

## CANDIDATE A

**Fidelity violations:**

1. **Dropped baseline figure.** Summary: *"Newsletter open rate declined to 35%"* — Email: *"Newsletter open rate dropped from 42% to 35%"*. The 42% baseline is omitted, so the CEO cannot see the size of the drop (7 points). The rubric explicitly counts a baseline figure as a number.
2. **Dropped dependency/blocker.** Summary: *"Product video campaign delayed until November"* — Email: *"delayed until November, pending legal approval; creative assets are ready."* The legal-approval dependency is the actionable blocker (the thing the CEO could unblock) and it is absent entirely. Also drops that creative is ready, i.e. that legal is the *only* thing gating it.
3. Minor: *"testing new subject lines scheduled"* loses the email's "next week," a time reference, though this is the weakest of the three.

**Test 1: FAIL** (dropped baseline number; dropped the legal-approval blocker).
Test 2: not assessed. (It would otherwise pass on length — ~40 words — 

## Dataset `Module1Output` + per-row Conciseness judge

The cold-start prompt below produced 20 rows (run once with `gpt-5.5`, saved verbatim as `fixtures/module1-starter-dataset.csv` with the model's first-pass labels). The cell after it attaches a per-row judge that returns 1 (good) or 0 (bad) under the same rubric, runs it over the 20 rows plus Version A and B, and writes `fixtures/module1-judge-results.csv`. Disagreements between the judge and the first-pass labels are the curation queue.

In [5]:
COLD_START_PROMPT = (
    "Generate 20 example rows for evaluating executive email-summary quality. Each row: (1) a short internal "
    "business email to a CEO, 60 to 120 words, with at least one metric, one risk or delay, and one deadline; "
    "(2) a candidate summary of it; (3) a first-pass label \"good\" or \"bad\"; (4) a one-line reason. Make about "
    "half \"good\": concise, every metric and deadline preserved, no invented facts. Make the other half \"bad\" in "
    "varied ways: drops the deadline, changes a number, adds a claim not in the email, or is padded past 100 words. "
    "Vary the domains (marketing, sales, engineering, finance, ops). Return a markdown table with columns id, email, "
    "summary, label, reason."
)

# Regenerate only if you want a fresh starter set; the committed CSV is the run of 2026-09-17.
# rows_md = openai_client.chat.completions.create(model="gpt-5.5", messages=[{"role": "user", "content": COLD_START_PROMPT}]).choices[0].message.content

with open(FIXTURES / "module1-starter-dataset.csv", newline="") as f:
    dataset = list(csv.DictReader(f))
dataset += [
    {"id": "A", "email": EMAIL, "summary": output_a, "first_pass_label": "", "first_pass_reason": "Version A, Concise"},
    {"id": "B", "email": EMAIL, "summary": output_b, "first_pass_label": "", "first_pass_reason": "Version B, Narrative"},
]
print(len(dataset), "rows in Module1Output")

22 rows in Module1Output


In [6]:
class Verdict(BaseModel):
    fidelity_pass: bool
    brevity_pass: bool
    score: int
    reason: str


def judge(email: str, summary: str) -> dict:
    """Conciseness LLM-as-a-Judge: 1 = concise and faithful under the rubric, 0 = not."""
    prompt = f"""You are grading one candidate summary of a source email against a strict rubric.

RUBRIC:
{GOLDEN_CRITERIA}

SOURCE EMAIL:
{email}

CANDIDATE SUMMARY:
{summary}

Return: fidelity_pass (Test 1), brevity_pass (Test 2; false if Test 1 failed), score (1 if both pass else 0),
and reason (one sentence naming the decisive violation or, for a pass, what it preserved)."""
    r = claude.messages.parse(
        model=JUDGE_MODEL,
        max_tokens=16000,
        messages=[{"role": "user", "content": prompt}],
        output_format=Verdict,
    )
    return r.parsed_output.model_dump()


results = []
for row in dataset:
    v = judge(row["email"], row["summary"])
    results.append({**row, "judge_score": v["score"], "judge_fidelity": v["fidelity_pass"],
                    "judge_brevity": v["brevity_pass"], "judge_reason": v["reason"]})
    fp = row["first_pass_label"] or "-"
    flag = "" if fp == "-" or (fp == "good") == (v["score"] == 1) else "  <-- disagrees with first-pass label"
    print(f'{row["id"]:>2}  first-pass={fp:<4}  judge={v["score"]}  {v["reason"]}{flag}')

with open(FIXTURES / "module1-judge-results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    w.writeheader(); w.writerows(results)

labeled = [r for r in results if r["first_pass_label"]]
agree = sum((r["first_pass_label"] == "good") == (r["judge_score"] == 1) for r in labeled)
print(f"\nJudge agrees with first-pass label on {agree}/{len(labeled)} rows.")
print(f"Version A score: {results[-2]['judge_score']}   Version B score: {results[-1]['judge_score']}")

 1  first-pass=good  judge=1  Preserves the 18% lift, the legal-review blocker, and the explicit Oct 4 approval ask in 24 words with no additions or softening.


 2  first-pass=good  judge=1  It keeps the $4.2M pipeline figure and the Sep 30 deadline, and names the procurement slowdown blocker at actionable level (could push signatures out) without softening it, in 23 words with the ask stated explicitly.


 3  first-pass=good  judge=1  It keeps the 0.7% peak error rate, the security review blocker on the production release, and the explicit Nov 15 go/no-go deadline, all in well under 60 words.


 4  first-pass=good  judge=1  It keeps the $1.1M monthly burn, the Aug 9 sign-off deadline, and the late-vendor-invoice risk to the August close without softening or inventing anything, in 26 actionable words.


 5  first-pass=good  judge=1  Preserves the 14,000/day figure, the carrier capacity slip risk during the holiday ramp, and the explicit Dec 12 approval ask in 23 words.


...

18  first-pass=bad   judge=0  It drops the 87% completion rate baseline figure, replacing it with the vague "is progressing," which violates the no-dropped-numbers rule despite correctly keeping the facilitator blocker and Feb 20 ask.


19  first-pass=bad   judge=0  It invents material the email never states — that 73% "gives a reasonably broad view of account health" and that this is an "executive-level alignment issue involving customer trust, renewal protection, product reliability, finance impact" — and buries the Apr 5 decision deadline in that padding at over 100 words.


20  first-pass=bad   judge=0  It fabricates that coworking space is already reserved, drops the permitting-delay risk and the Jun 3 approval deadline/ask, leaving the CEO falsely reassured.


 A  first-pass=-     judge=0  It drops the 42% baseline for the newsletter open rate and omits the legal-approval dependency behind the video delay, leaving the CEO unaware of the blocker and the true size of the drop.


 B  first-pass=-     judge=0  It softens the risk by recasting the delayed product video campaign as "on track to launch in November" and adds unsupported optimism ("promising results," "position us well for sustained growth"), while also omitting that next week is when subject lines get tested.

Judge agrees with first-pass label on 20/20 rows.
Version A score: 0   Version B score: 0
